# 03. Гипотеза повторной покупки и дизайн эксперимента

Вопрос: увеличит ли одно напоминание через 14 суток после первого заказа вероятность следующей покупки? Это **план будущего теста**, а не проведённый эксперимент.

В RFM есть 269 недавних однократных клиентов. Для эксперимента уточняем точку входа: ровно через 14 суток после первой покупки, если до этого момента существует только один заказ. Такая очередь набирается постепенно. Число 269 на одном срезе не равно размеру этой очереди.

Сначала оценим исторический аналог, затем рассчитаем необходимую выборку. Контактов, разрешения на коммуникации, подтверждённого статуса оплаты и данных кампании в UCI нет.

In [1]:
from pathlib import Path
from contextlib import closing
from statistics import NormalDist
import json
import math
import sqlite3
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'docs/01_project_brief.md').is_file())
DB = ROOT / 'data/processed/retail.sqlite'
TABLES = ROOT / 'reports/tables'
CUTOFF = pd.Timestamp('2011-12-01')
NORMAL = NormalDist()
ALPHA, TARGET_POWER = 0.05, 0.80
with closing(sqlite3.connect(DB.as_uri() + '?mode=ro', uri=True)) as conn:
    historical = pd.read_sql_query(
        (ROOT / 'sql/07_day14_experiment_baseline.sql').read_text(), conn,
        params={'cutoff': str(CUTOFF)})
    orders = pd.read_sql_query('SELECT * FROM customer_orders', conn)
orders['order_date'] = pd.to_datetime(orders['order_date'])
for column in ['first_order_date', 'anchor_date', 'observation_end', 'next_order_after_anchor']:
    historical[column] = pd.to_datetime(historical[column])
print('Срез:', CUTOFF.date(), '| клиентов в истории:', len(historical))

Срез: 2011-12-01 | клиентов в истории: 4293


## 1. Историческая точка отбора

`anchor = first_order + 14 суток`. Включение использует только данные до anchor, включая границу. Измерение покупки — строго после anchor и не позднее anchor + 30 суток. Одновременные дополнительные счета считаются дополнительными заказами при отборе; клиент с ними не соответствует F = 1.

Первая наблюдаемая покупка должна быть с января 2011 года: декабрьская когорта может содержать старых клиентов. Для анализа исхода требуется `observation_end < cutoff`. У неполного окна исход отсутствует. Это приближение к будущей аудитории, поскольку полная история до выгрузки и контактность неизвестны.

In [2]:
# Независимая проверка SQL через pandas для каждого клиента.
first = orders.groupby('customer_id')['order_date'].min().rename('first_order_date').reset_index()
first['anchor_date'] = first['first_order_date'] + pd.Timedelta(days=14)
first['observation_end'] = first['anchor_date'] + pd.Timedelta(days=30)
joined = orders.merge(first, on='customer_id', validate='many_to_one')
counts = joined.loc[joined['order_date'] <= joined['anchor_date']].groupby('customer_id').size()
next_dates = joined.loc[joined['order_date'] > joined['anchor_date']].groupby('customer_id')['order_date'].min()
reference = first.set_index('customer_id')
reference['invoices_at_anchor'] = counts
reference['next_order_after_anchor'] = next_dates
reference['eligible_at_day14'] = (
    reference['first_order_date'].ge('2011-01-01')
    & reference['anchor_date'].lt(CUTOFF) & reference['invoices_at_anchor'].eq(1)).astype(int)
reference['full_followup'] = reference['observation_end'].lt(CUTOFF).astype(int)
reference['repeat_in_next_30d'] = (
    reference['next_order_after_anchor'].le(reference['observation_end']).astype(float))
reference.loc[reference['full_followup'].eq(0), 'repeat_in_next_30d'] = np.nan
sql = historical.set_index('customer_id').sort_index()
reference = reference.sort_index()
pd.testing.assert_frame_equal(sql[reference.columns], reference, check_dtype=False)
assert historical.loc[historical['full_followup'].eq(0), 'repeat_in_next_30d'].isna().all()
historical.to_csv(TABLES / 'day14_historical_customers.csv', index=False)

eligible = historical.loc[historical['eligible_at_day14'].eq(1)].copy()
eligible['first_cohort'] = eligible['first_order_date'].dt.to_period('M').astype(str)
complete = eligible.loc[eligible['full_followup'].eq(1)].copy()
events, denominator = int(complete['repeat_in_next_30d'].sum()), len(complete)
baseline = events / denominator
cohorts = eligible.groupby('first_cohort').agg(
    eligible_at_day14=('customer_id', 'size'), complete_followup=('full_followup', 'sum'),
    repeat_customers=('repeat_in_next_30d', 'sum'))
cohorts.loc[cohorts['complete_followup'].eq(0), 'repeat_customers'] = np.nan
cohorts['repeat_pct'] = 100 * cohorts['repeat_customers'].div(cohorts['complete_followup'].replace(0, np.nan))
cohorts.to_csv(TABLES / 'day14_historical_cohorts.csv')
print(cohorts.round(2).to_string())
print(f'Исторический аналог: {events} / {denominator} = {100*baseline:.2f}%')

              eligible_at_day14  complete_followup  repeat_customers  repeat_pct
first_cohort                                                                    
2011-01                     376                376              82.0       21.81
2011-02                     349                349              55.0       15.76
2011-03                     416                416              61.0       14.66
2011-04                     275                275              53.0       19.27
2011-05                     251                251              38.0       15.14
2011-06                     221                221              38.0       17.19
2011-07                     167                167              32.0       19.16
2011-08                     162                162              33.0       20.37
2011-09                     275                275              61.0       22.18
2011-10                     317                174              32.0       18.39
2011-11                     

## 2. Неопределённость исторического ориентира

Интервал Уилсона описывает биномиальную неопределённость исторической доли. Он не учитывает различия месяцев, неизвестную контактность и изменение поведения в будущем. Это не прогноз контрольной группы.

In [3]:
def wilson_interval(successes, total, alpha=0.05):
    p = successes / total
    z = NORMAL.inv_cdf(1 - alpha / 2)
    scale = 1 + z*z/total
    center = (p + z*z/(2*total)) / scale
    radius = z * math.sqrt(p*(1-p)/total + z*z/(4*total*total)) / scale
    return center - radius, center + radius

baseline_ci = wilson_interval(events, denominator)
print('95%% Wilson CI: %.2f%%–%.2f%%' % tuple(100*v for v in baseline_ci))

95% Wilson CI: 16.77%–19.70%


## 3. Мощность и необходимая выборка

Двусторонний тест двух независимых долей, α = 0,05, мощность 80%, распределение 1:1. Нормальное приближение: объединённая дисперсия при нулевой гипотезе и раздельная при альтернативе. Предполагаем независимость клиентов. Калькулятор реализован на стандартной библиотеке; его метод соответствует [описанию statsmodels](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.power_proportions_2indep.html).

Абсолютный эффект +5 п.п. — сценарий планирования, **не ожидаемый результат и не обоснованный экономикой порог**. Дополнительно рассмотрим +3 и +10 п.п., а также контрольные доли 10% и 20%.

In [4]:
def proportion_power(p0, uplift, n_per_group, alpha=0.05):
    p1 = p0 + uplift
    if not (0 < p0 < 1 and 0 <= uplift < 1-p0 and n_per_group > 0):
        raise ValueError('Нужны допустимые доли и положительная выборка.')
    average = (p0+p1)/2
    null_se = math.sqrt(2*average*(1-average)/n_per_group)
    alternative_se = math.sqrt((p0*(1-p0)+p1*(1-p1))/n_per_group)
    boundary = NORMAL.inv_cdf(1-alpha/2) * null_se
    return (NORMAL.cdf((-boundary-uplift)/alternative_se)
            + 1 - NORMAL.cdf((boundary-uplift)/alternative_se))

def required_n(p0, uplift, target=TARGET_POWER):
    low, high = 1, 2
    while proportion_power(p0, uplift, high) < target:
        high *= 2
    while low < high:
        mid = (low + high) // 2
        if proportion_power(p0, uplift, mid) >= target:
            high = mid
        else:
            low = mid + 1
    return low

rows = []
for label, p0 in [('10%', 0.10), ('Исторический аналог', baseline), ('20%', 0.20)]:
    for effect in [0.03, 0.05, 0.10]:
        n = required_n(p0, effect)
        rows.append({'baseline_scenario': label, 'baseline_pct': 100*p0,
                     'uplift_pp': 100*effect, 'n_per_group': n, 'total_n': 2*n,
                     'achieved_power_pct': 100*proportion_power(p0, effect, n)})
sample_sizes = pd.DataFrame(rows)
sample_sizes.to_csv(TABLES / 'experiment_sample_sizes.csv', index=False)
assert abs(proportion_power(baseline, 0, 134)-ALPHA) < 1e-12
for row in rows:
    assert proportion_power(row['baseline_pct']/100, row['uplift_pp']/100, row['n_per_group']) >= TARGET_POWER
    if row['n_per_group'] > 1:
        assert proportion_power(row['baseline_pct']/100, row['uplift_pp']/100, row['n_per_group']-1) < TARGET_POWER
print(sample_sizes.round(2).to_string(index=False))

  baseline_scenario  baseline_pct  uplift_pp  n_per_group  total_n  achieved_power_pct
                10%         10.00        3.0         1774     3548               80.00
                10%         10.00        5.0          686     1372               80.02
                10%         10.00       10.0          199      398               80.01
Исторический аналог         18.19        3.0         2758     5516               80.01
Исторический аналог         18.19        5.0         1030     2060               80.03
Исторический аналог         18.19       10.0          279      558               80.08
                20%         20.00        3.0         2943     5886               80.01
                20%         20.00        5.0         1094     2188               80.01
                20%         20.00       10.0          294      588               80.11


## 4. Достаточно ли 269 клиентов?

Разделение даёт 134 и 135 клиентов. Для простого консервативного расчёта используем по 134. Все 269 здесь — **только условный объём**: это другой RFM-срез, не готовая выборка на 14-е сутки. Разрешение на коммуникацию дополнительно уменьшит доступную аудиторию.

Ниже оцениваем мощность этого объёма и эффект, который он способен обнаружить с мощностью 80%. Это расчёт при выбранных допущениях, а не результат кампании.

In [5]:
snapshot_n = 269
small_n = snapshot_n // 2
small_power = proportion_power(baseline, 0.05, small_n)
low, high = 0.0, 1-baseline-1e-8
for _ in range(80):
    mid = (low+high)/2
    if proportion_power(baseline, mid, small_n) >= TARGET_POWER:
        high = mid
    else:
        low = mid
small_mde = high
n_for_5pp = required_n(baseline, 0.05)
print(f'269 клиентов: мощность для +5 п.п. около {100*small_power:.1f}%')
print(f'Обнаружимый эффект при 80% мощности: около +{100*small_mde:.1f} п.п.')
print(f'Для +5 п.п.: {n_for_5pp} на группу, всего {2*n_for_5pp}')

# Только сценарии скорости набора; CRM-контактность и будущий поток неизвестны.
duration = pd.DataFrame({'eligible_contactable_customers_per_month': [100, 250, 500]})
duration['enrollment_months'] = 2*n_for_5pp / duration['eligible_contactable_customers_per_month']
duration['followup_days_after_last_enrollment'] = 30
duration.to_csv(TABLES / 'experiment_duration_scenarios.csv', index=False)
print(duration.round(2).to_string(index=False))

269 клиентов: мощность для +5 п.п. около 17.2%
Обнаружимый эффект при 80% мощности: около +14.9 п.п.
Для +5 п.п.: 1030 на группу, всего 2060
 eligible_contactable_customers_per_month  enrollment_months  followup_days_after_last_enrollment
                                      100              20.60                                   30
                                      250               8.24                                   30
                                      500               4.12                                   30


## 5. Вывод для решения

Сначала проверить CRM-историю, статус первого заказа и доступность контакта; затем оценить контрольную долю на тех же правилах и поток участников. При малом потоке — пилот доставки и сбора событий; он не даёт надёжной проверки небольшого эффекта. Для подтверждающего эксперимента заранее зафиксировать MDE, выборку, период и правило анализа.

Основная метрика будущего теста — покупка за 30 дней **после назначения группы**. Анализ по всем назначенным клиентам (ITT), включая недоставленные письма. Экспериментальные исходы в этом ноутбуке отсутствуют. Полный протокол — [docs/02_experiment_plan.md](../docs/02_experiment_plan.md).

In [6]:
summary = {
    'analysis_cutoff': str(CUTOFF.date()), 'anchor_days_after_first_order': 14,
    'followup_days_after_anchor': 30, 'historical_complete_customers': denominator,
    'historical_repeat_customers': events, 'historical_repeat_pct': 100*baseline,
    'historical_wilson_95_pct': [100*v for v in baseline_ci],
    'alpha': ALPHA, 'target_power': TARGET_POWER, 'allocation': '1:1',
    'planning_uplift_pp': 5, 'required_n_per_group_for_5pp': n_for_5pp,
    'required_total_n_for_5pp': 2*n_for_5pp,
    'illustrative_269_power_for_5pp_pct': 100*small_power,
    'illustrative_269_mde_pp': 100*small_mde,
    'experiment_conducted': False,
    'checks': ['day14_sql_pandas_all_customers', 'unobserved_outcomes_null',
               'power_at_null', 'minimum_integer_sample_size'],
}
(ROOT / 'reports/stage5_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2)+'\n')
print('Сохранены исторический ориентир и сценарии планирования. Эксперимент не проводился.')

Сохранены исторический ориентир и сценарии планирования. Эксперимент не проводился.
